# Mapping ARCHS4 Tissue Categories

The objective of this notebook is to resolve the tissue annotations of the ARCHS4 data using UBERON IDs, which are also used by single-cell collection CellxGene to map to tissue categories. This would allow us to establish a direct link between ARCHS4 metadata fields and CellxGene tissue categories through UBERON IDs.

In [1]:
#Imports
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path
import os
import json
import re
import matplotlib.pyplot as plt
import scipy.sparse as sp
from pathlib import Path
import re
import h5py

## Load Data

In [2]:
# Original ARCHS4
BULK_DIR = Path("/cluster/work/boeva/rquiles/data/ARCHS4-data/data/processed/archs4")
EXPRESSION_H5 = BULK_DIR / "expression.h5"
GENE_LIST_TXT = BULK_DIR / "gene_list.txt"
METADATA_CSV = BULK_DIR / "metadata.csv"

metadata = pd.read_csv(METADATA_CSV)

In [3]:
metadata.head(3)

,sample_id,geo_accession,series_id,platform_id,instrument,organism,characteristics,extract_protocol,source_name,title,taxid,type,total_counts,n_detected_genes
0,GSM1000981,GSM1000981,GSE29282,GPL15433,Illumina HiSeq 1000,Homo sapiens,"treatment: siNT,cell line: OCI-LY1",ChIP-seq libraries were prepared using the Ill...,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R1,9606,SRA,70707856.0,32704
1,GSM1000982,GSM1000982,GSE29282,GPL15433,Illumina HiSeq 1000,Homo sapiens,"treatment: siNT,cell line: OCI-LY1",ChIP-seq libraries were prepared using the Ill...,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R2,9606,SRA,58297056.0,32070
2,GSM1000983,GSM1000983,GSE29282,GPL15433,Illumina HiSeq 1000,Homo sapiens,"treatment: siNT,cell line: OCI-LY1",ChIP-seq libraries were prepared using the Ill...,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R3,9606,SRA,65223632.0,32713


In [4]:
# CellxGene Data
CELLXGENE_DIR = Path("/cluster/work/boeva/atheus/data_cellxgene")
FILE = CELLXGENE_DIR / "partition_100.h5ad"

adata_sc = sc.read_h5ad(FILE, backed="r")

In [5]:
adata_sc.obs.head(3)

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars,n_counts
0,35111238,4a5b00e0-1ba3-4fd4-af89-d3512eb20720,10x 3' v3,EFO:0009922,hematopoietic precursor cell,CL:0008001,41-year-old stage,HsapDv:0000135,normal,PATO:0000461,...,UBERON:0000178,tissue,blood,UBERON:0000178,1580.0,940,1.680851,3.906555,28560,1580.0
1,35111239,4a5b00e0-1ba3-4fd4-af89-d3512eb20720,10x 3' v3,EFO:0009922,hematopoietic precursor cell,CL:0008001,30-year-old stage,HsapDv:0000124,normal,PATO:0000461,...,UBERON:0000178,tissue,blood,UBERON:0000178,2504.0,1121,2.233720,16.316755,28560,2504.0
2,35111240,4a5b00e0-1ba3-4fd4-af89-d3512eb20720,10x 3' v3,EFO:0009922,hematopoietic precursor cell,CL:0008001,36-year-old stage,HsapDv:0000130,normal,PATO:0000461,...,UBERON:0000178,tissue,blood,UBERON:0000178,1771.0,886,1.998871,8.496044,28560,1771.0


## ARHS4 metadata --> UBERON IDs

Steps:
1) Find key:value pairs in characteristic fields, determine curation queue.
2) Exact matching to UBERON labels and synonims
3) Fuzzy matching

### 1) Find key:value pairs and determine curation queue

First, we split the characteristics field to generate new fields denoted by the key:value pairs found inside that field. We then assess the coverage that these fields provide.

In [6]:
# characteristics are joined by "Xx-xX" or by a comma followed by a new "key:"
SPLIT = re.compile(r"Xx-xX|,(?=\s*[^,:]{1,60}:)")

# explicit key lists, in priority order (first key a sample has wins)
TISSUE_KEYS = [
    "tissue", "tissue type", "tissuetype", "organ", "source organ", "organ site", "source tissue",
    "tissue source", "tissue origin", "tissue of origin", "tissue (isolated from)", "body site",
    "anatomical site", "anatomic region", "sampling site", "biopsy site", "biopsy location",
    "organ/tissue", "tissue region", "tissue subtype",
    "tissue/celltype", "tissue/cell type", "tissue/cell type source",
    "cell line origin",                     # tissue a cell line came from; these samples are still flagged has_cell_line
]
CELL_TYPE_KEYS = [
    "cell type", "celltype", "curated-cell-type", "cell type sorted", "sample cell type", "source cell type",
    "endocrine cell-type", "inferred cell type", "cell type or stage", "cell population", "cell subtype",
    "cell types", "cell line/type",
]
CELL_LINE_KEYS = [
    "cell line", "cell line name", "cellline", "cell line id", "inhouse cell line name",
    "source cell line", "originating cell line", "parental cell line", "cell line background",
]
FALLBACK_TISSUE_KEYS = ["organ system"]     # too coarse for UBERON; only used when nothing else is available

# reviewed and deliberately ignored; hidden from the "consider adding" printout
SKIP_KEYS = {
    "organism", "organisms", "organ donation death type", "organoidid", "tissue preparation",
    "cell type surface markers", "age at biopsy", "tissuestoragetime", "tissue collection",
    "tissue composition", "tissue disease state", "cancer cell line composition",
    "cell line group", "sample cell type cat",
}
SUGGEST = re.compile(r"tissue|organ|body.?site|anatom|biopsy|cell.?type|cell.?line", re.I)   # only to suggest extra keys (case-insensitive)

NA = {"", "nan", "none", "na", "n/a", "not applicable", "missing", "unknown", "not available"}

# String normalizer function
def norm(s) -> str:
    s = re.sub(r"\s+", " ", re.sub(r"_+", " ", str(s).lower())).strip(" ,;")
    return "" if s in NA else s

# ---------------- 1) load ----------------
metadata = pd.read_csv(METADATA_CSV, dtype=str, low_memory=False)
meta = metadata.set_index("sample_id")
meta.index = meta.index.astype(str)
assert meta.index.is_unique, "duplicated sample_id in metadata.csv"
print(f"{len(meta):,} samples")

# ---------------- 2) characteristics -> key: value pairs ----------------
rows = []
for sid, txt in meta["characteristics"].dropna().items():
    for part in SPLIT.split(txt):
        if ":" in part:
            k, v = part.split(":", 1)
            k, v = norm(k), norm(v)
            if k and v:
                rows.append((sid, k, v))

# stores the key-value pairs associated to the samples that had them
kv = pd.DataFrame(rows, columns=["sample_id", "key", "value"]).drop_duplicates(["sample_id", "key"])
print(f"{len(kv):,} key: value pairs")

# Number of samples for each given key, sorted
key_counts = kv.groupby("key")["sample_id"].nunique().sort_values(ascending=False)
print("\nmost common keys:\n", key_counts.head(30).to_string())

listed = set(TISSUE_KEYS + CELL_TYPE_KEYS + CELL_LINE_KEYS + FALLBACK_TISSUE_KEYS) | SKIP_KEYS
suggest = key_counts[[k for k in key_counts.index if SUGGEST.search(k) and k not in listed]]
print("\nkeys NOT in the lists that look relevant (consider adding):\n", suggest.head(25).to_string())

def pick(keys):
    """Value (and key) of the first key in `keys` that each sample has (e.g. only one tissue key per sample is recovered)"""
    rank = {k: i for i, k in enumerate(keys)}
    sub = kv[kv["key"].isin(rank)].assign(r=lambda d: d["key"].map(rank)).sort_values(["sample_id", "r"])
    sub = sub.drop_duplicates("sample_id").set_index("sample_id") # mantains only values from top keys, as they are sorted
    return sub["value"], sub["key"]

tissue_v, tissue_k = pick(TISSUE_KEYS)
celltype_v, _      = pick(CELL_TYPE_KEYS)
cellline_v, _      = pick(CELL_LINE_KEYS)
fallback_v, _      = pick(FALLBACK_TISSUE_KEYS)

# ---------------- 3) per-sample table ----------------
# all dataframes sharing the index ensures the mapping correct throughout
samples = pd.DataFrame(index=meta.index)
samples.index.name = "sample_id"
samples["series_id"]       = meta["series_id"]
samples["source_name"]     = meta["source_name"].map(norm)
samples["tissue_key"]      = tissue_k
samples["tissue_value"]    = tissue_v
samples["cell_type_value"] = celltype_v
samples["cell_line_value"] = cellline_v
samples["tissue_fallback"] = fallback_v      # e.g. "nervous system"; use only if tissue_value and source_name fail
samples["title"]           = meta["title"].map(norm)
samples["characteristics"] = meta["characteristics"].map(norm)
samples = samples.replace("", np.nan)
samples["has_cell_line"]   = samples["cell_line_value"].notna()

print("\nfield presence:")
for c in ["source_name", "tissue_value", "cell_type_value", "cell_line_value", "tissue_fallback"]:
    print(f"  {c:<16} {samples[c].notna().mean():6.1%}")
print(f"  no tissue value and no source name: {(samples['tissue_value'].isna() & samples['source_name'].isna()).sum():,}")
print("  tissue_value by key:\n" + samples["tissue_key"].value_counts().to_string())

# ---------------- 4) unique strings ranked by coverage ----------------
# Table of unique strings of a given column ordered by the amount of samples where they appear
def ranked(col):
    s = samples[col].dropna() # Series whose values are strings 
    g = pd.DataFrame({
        "n_samples": s.groupby(s).size(), # group using the unique values in s
        "n_series": samples.loc[s.index, "series_id"].groupby(s).nunique(),
        "example_id": s.groupby(s).apply(lambda x: x.index[0]),
        "frac_cell_line": samples.loc[s.index, "has_cell_line"].groupby(s).mean().round(2),
    }).sort_values("n_samples", ascending=False)
    g["cum_frac_samples"] = g["n_samples"].cumsum() / len(samples)
    g.index.name = "string"
    return g

queues = {}
for col in ["source_name", "tissue_value", "cell_type_value"]:
    queues[col] = r = ranked(col)
    cov = ", ".join(f"top {n}: {r['cum_frac_samples'].iloc[min(n, len(r)) - 1]:.1%}" for n in (100, 500, 1000, 5000))
    print(f"\n{col}: {len(r):,} unique strings | {cov}")
    print(r.head(10).to_string())

# ---------------- 5) combined queue: tissue value if present, else source name ----------------
prim = samples["tissue_value"].fillna(samples["source_name"])
field = pd.Series(np.where(samples["tissue_value"].notna(), "tissue", "source_name"), index=samples.index)
queue = (pd.DataFrame({"string": prim, "field": field, "series_id": samples["series_id"],
                       "cell_line": samples["has_cell_line"]})
         .dropna(subset=["string"])
         .groupby(["field", "string"])
         .agg(n_samples=("series_id", "size"), n_series=("series_id", "nunique"),
              frac_cell_line=("cell_line", "mean"))
         .sort_values("n_samples", ascending=False)
         .reset_index())
queue["frac_cell_line"] = queue["frac_cell_line"].round(2)
queue["cum_frac_samples"] = queue["n_samples"].cumsum() / len(samples)
queues["combined"] = queue

print(f"\ncombined queue: {len(queue):,} (field, string) pairs")
for n in (200, 500, 1000, 2000, 5000):
    print(f"  top {n:>5}: {queue['cum_frac_samples'].iloc[min(n, len(queue)) - 1]:.1%} of samples")
non_cl = queue[queue["frac_cell_line"] < 0.5]
n_non_cl = int((~samples["has_cell_line"]).sum())
print(f"  excluding cell-line-dominated strings, top 1000 cover "
      f"{non_cl['n_samples'].head(1000).sum() / n_non_cl:.1%} of the {n_non_cl:,} non-cell-line samples")
queue.head(20)

719,432 samples
3,171,089 key: value pairs

most common keys:
 key
cell type             316973
tissue                253350
cell line             205047
treatment             201709
age                    91498
sex                    82933
genotype               80251
gender                 45634
time                   44371
disease state          39295
patient id             27141
batch                  24729
well                   23799
donor                  22631
individual             22453
disease                22291
group                  20780
condition              19937
diagnosis              18634
time point             18118
sample type            17516
timepoint              16422
donor id               16267
subject id             15710
plate                  14829
replicate              14094
genotype/variation     13722
race                   12095
passage                11135
ethnicity              10721

keys NOT in the lists that look relevant (consider adding):
 k

,field,string,n_samples,n_series,frac_cell_line,cum_frac_samples
0,tissue,blood,22841,482,0.09,0.031749
1,tissue,whole blood,15763,172,0.02,0.053659
2,tissue,skin,9438,260,0.23,0.066778
3,source_name,lcl-derived ipsc,8682,3,0.00,0.078846
4,tissue,peripheral blood,8507,374,0.08,0.090670
5,tissue,bone marrow,7936,242,0.09,0.101701
6,source_name,mcf7 cells,7296,26,1.00,0.111842
7,tissue,tumor,6686,117,0.35,0.121136
8,source_name,pbmc,6448,152,0.05,0.130098
9,tissue,pbmc,6319,126,0.00,0.138882


### 2) Exact Matching

Matches strings determined before to the UBERON ontology in the following steps (the first step that fires decides the result, and order reflects confidence):

1) **Clean the text**: fix encoding errors, strip accents, correct known typos.
2) **Generate variants**: strip prefixes/suffixes, swap adjectives for nouns, drop plurals, strip cell/neuron suffixes, add “tissue”/”organ”, remove parentheses, and remove tumor words together with “adjacent/normal/uninvolved” words. The text inside parentheses (4+ characters) is kept apart as the weakest variant.
3) **Manual decisions**: if any variant is in the hand-curated MANUAL table, use it.
4) **Tumor names**: if a whole variant is a known tumor name, map it to its organ.
5) **Exclusions**: return no ID, with the reason, for non-tissues, control RNA, xenografts, cell line names and in vitro systems.
6) **Special rules**: hematopoietic, nasal epithelium, nasopharynx and blood patterns (blood only if no other body site is mentioned).
7) **Tumor abbreviations**: if any single word is a known tumor abbreviation, map it to its organ.
8) **Parentheses**: MANUAL and tumor names on the text inside parentheses (“primary tumor (colorectum)”), only now, so “cell culture (blood)” stays in vitro.
9) **Ontology lookup**: try official labels and EXACT synonyms on all variants (parenthesis text included), then RELATED synonyms (parenthesis text and appended “tissue”/”organ” excluded: “frontal” + “ organ” is a RELATED synonym of the pineal body). Variants that are generic NOT_A_TISSUE words (“tissue”, “cortex”, “epithelium”) are never looked up. If a string matches several terms, report “ambiguous”. Tags mark variant matches (+variant) and tumor context (+tumor).
10) **Organ words**: look for organ keywords (plurals included). One distinct organ gives a match; several give “ambiguous organ words”.
11) **Generic leftovers**: “not a tissue” if any variant is a NOT_A_TISSUE word (“ffpe tumor” -> “tumor”) or the string is a sample code / protocol description (SAMPLE_CODE: “bc05 single-cell rna-seq”, “mm34em”).
12) **No match**: otherwise report “no match”.

In [7]:
# obonet library for ontology parsing
import re, obonet
from collections import defaultdict
import networkx as nx

UBERON_OBO = "/cluster/work/boeva/rquiles/ontologies/uberon-basic.obo"
N_REVIEW = 2000          # top strings reviewed by hand

In [8]:
# Parse UBERON ontology
g = obonet.read_obo(UBERON_OBO)                 # obsolete terms are skipped by default
SYN = re.compile(r'^"(.*)"\s+(EXACT|RELATED|NARROW|BROAD)\b')   # obonet keeps synonyms as raw OBO strings
DEF = re.compile(r'^"(.*)"')
_ANC_GRAPH = {}

# Term dictionary of UBERON IDs and label names
terms = {n: {"name": d["name"]} for n, d in g.nodes(data=True)
         if n.startswith("UBERON:") and d.get("name")}

# Dictionary of 3 dictionaries of sets (everything lowecase to ease matching):
# label: name --> {UBERON_IDs}
# exact: synonym --> {UBERON_IDs}
# related : synonym --> {UBERON_IDs}
lookup = {"label": defaultdict(set), "exact": defaultdict(set), "related": defaultdict(set)}
for n, d in g.nodes(data=True):
    if n not in terms:
        continue
    lookup["label"][d["name"].lower()].add(n)
    for s in d.get("synonym", []):
        m = SYN.match(s)
        if m and m.group(2) in ("EXACT", "RELATED"):
            lookup[m.group(2).lower()][m.group(1).lower()].add(n)
print(f"{len(terms):,} UBERON terms")

15,045 UBERON terms


In [11]:
def uid(label):
    """UBERON id for an exact label or EXACT synonym (fails loudly if missing or ambiguous)."""
    ids = lookup["label"].get(label) or lookup["exact"].get(label)
    assert ids and len(ids) == 1, f"{label!r}: {ids}"
    return next(iter(ids))

def term_info(g, tid, relations=("is_a", "part_of"), max_ancestors=25):
    """Print what UBERON says about one term. In obonet, edges point child -> parent, keyed by relation."""
    if tid not in g:
        print(f"{tid} not found (obsolete or wrong id)"); return
    name = lambda t: g.nodes[t].get("name", "?") if t in g else "?"
    d = g.nodes[tid]
    print(f"{tid}  {d.get('name')}")

    m = DEF.match(d.get("def", ""))
    if m:
        print(f"  definition: {m.group(1)}")

    syns = [SYN.match(s).groups() for s in d.get("synonym", []) if SYN.match(s)]
    for scope in ("EXACT", "RELATED", "NARROW", "BROAD"):
        vals = [s for s, sc in syns if sc == scope]
        if vals:
            print(f"  {scope.lower()} synonyms: {', '.join(vals)}")

    print("  parents:")
    for _, parent, rel in g.out_edges(tid, keys=True):
        print(f"    {rel:<10} {parent}  {name(parent)}")

    children = [(c, rel) for c, _, rel in g.in_edges(tid, keys=True)]
    print(f"  children: {len(children)}"
          + (": " + ", ".join(f"{name(c)} ({rel})" for c, rel in children[:10]) if children else ""))

    # ancestors: follow only the chosen relations (graph built once per relation set)
    if relations not in _ANC_GRAPH:
        _ANC_GRAPH[relations] = nx.MultiDiGraph((u, v, k) for u, v, k in g.edges(keys=True) if k in relations)
    sub = _ANC_GRAPH[relations]
    anc = nx.descendants(sub, tid) if tid in sub else set()      # child -> parent edges, so descendants = ancestors
    dist = nx.single_source_shortest_path_length(sub, tid)
    print(f"  ancestors via {'/'.join(relations)}: {len(anc)} (closest first)")
    for a in sorted(anc, key=dist.get)[:max_ancestors]:
        print(f"    {a}  {name(a)}")

In [12]:
term_info(g, uid("blood plasma"))

UBERON:0001969  blood plasma
  definition: The liquid component of blood, in which erythrocytes are suspended.
  exact synonyms: blood plasm, portion of blood plasma
  broad synonyms: plasma, portion of plasma
  parents:
    is_a       UBERON:0000179  haemolymphatic fluid
    part_of    UBERON:0000178  blood
  children: 3: interstitial fluid (mutually_spatially_disjoint_with), blood serum (part_of), transudate (transformation_of)
  ancestors via is_a/part_of: 14 (closest first)
    UBERON:0000179  haemolymphatic fluid
    UBERON:0000178  blood
    UBERON:0002390  hematopoietic system
    UBERON:0006314  bodily fluid
    UBERON:0002193  hemolymphoid system
    UBERON:0004120  mesoderm-derived structure
    UBERON:0000467  anatomical system
    UBERON:0000463  organism substance
    UBERON:0000468  multicellular organism
    UBERON:0000001  gross anatomical part
    UBERON:0000061  anatomical structure
    UBERON:0000465  material anatomical entity
    UBERON:0010000  multicellular anato

In [13]:
# Manually crafted lists and maps
# ---------------- 2) manual seeds and rule lists ---------------

BLOOD, SKIN, HEMATO = uid("blood"), uid("skin of body"), uid("hematopoietic system")

# explicit string -> UBERON decisions (checked against every cleaned variant of a string)
MANUAL = {
    # blood and blood fractions
    "blood": BLOOD, "peripheral blood": BLOOD, "venous blood": BLOOD, "buffy coat": BLOOD,
    "mobilised peripheral blood": BLOOD, "mobilized peripheral blood": BLOOD,
    "pbmc": BLOOD, "pbmcs": BLOOD, "peripheral blood mononuclear cells": BLOOD,
    "peripheral blood mononuclear cell": BLOOD, "pbmcs from mtb exposed patients": BLOOD,
    "plasma": uid("blood plasma"), "plasma, extracted cell-free rna": uid("blood plasma"),
    "plasma extracellular vesicles": uid("blood plasma"),
    "serum": uid("blood serum"), "serum ev": uid("blood serum"),
    "cord blood": uid("umbilical cord blood"), "umbilical cord blood": uid("umbilical cord blood"),
    "cbmc": uid("umbilical cord blood"), "cbmcs": uid("umbilical cord blood"),
    "tumor educated platelets from blood": BLOOD,
    # hematopoietic, source not stated
    "blood or bone marrow": HEMATO, "bm": uid("bone marrow"),
    # lymph nodes
    "ln": uid("lymph node"), "mesenteric ln": uid("mesenteric lymph node"),
    # skin
    "psoriasis lesion": SKIN, "psoriatic skin": SKIN, "lesional skin": SKIN,
    "foreskin": uid("prepuce of penis"), "juvenile foreskin": uid("prepuce of penis"),
    # nose and airways
    "nasal": uid("nasal cavity"), "nasal lavage": uid("nasal cavity"), "saline nasal wash": uid("nasal cavity"),
    "nasal mucosa": uid("nasal cavity mucosa"),
    "bronchial epithelial cells": uid("epithelium of bronchus"), "bronchial epithelium": uid("epithelium of bronchus"),
    "brushing of bronchial epithelium": uid("epithelium of bronchus"),
    "cytologically normal bronchial airway epithelial cells": uid("epithelium of bronchus"),
    "small airway epithelium": uid("epithelium of bronchiole"),
    "small airway epithelium brushing": uid("epithelium of bronchiole"),
    "airway epithelium": uid("respiratory tract epithelium"), "airway epithelial cells": uid("respiratory tract epithelium"),
    "human airway epithelial cells": uid("respiratory tract epithelium"),
    "bronchoalveolar lavage": uid("lung"), "bronchoalveolar lavage (bal)": uid("lung"), "bal": uid("lung"),
    "sftpc+ cells": uid("lung"),                                       # alveolar type 2 cells
    "alveolar macrophages": uid("lung"),                               # from bronchoalveolar lavage
    # nervous system
    "hippocampus": uid("hippocampal formation"),
    "brain cortex": uid("cerebral cortex"), "cortex frontal": uid("frontal cortex"),
    "cortex motor lateral": uid("primary motor cortex"), "cortex motor medial": uid("primary motor cortex"),
    "cortex temporal": uid("temporal cortex"), "cortex occipital": uid("occipital cortex"),
    "ba9": uid("dorsolateral prefrontal cortex"),                     # Brodmann area 9
    "caudate": uid("caudate nucleus"), "ventral midbrain": uid("midbrain"),
    "meningioma": uid("meninx"), "retina punch": uid("retina"),
    "cns": uid("central nervous system"), "csf": uid("cerebrospinal fluid"),
    # digestive system
    "gut": uid("digestive tract"), "intestinal biopsy": uid("intestine"),
    "colon & rectum": uid("large intestine"), "crc": uid("large intestine"), "colorectal": uid("large intestine"),
    "oral mucosa": uid("mouth mucosa"),
    "islet": uid("islet of langerhans"), "islets": uid("islet of langerhans"),
    "islet cell": uid("islet of langerhans"), "islet cells": uid("islet of langerhans"),
    # reproductive system, placenta, embryo
    "endocervical cells": uid("endocervix"),
    "chorionic villous": uid("chorionic villus"), "chorionic villi": uid("chorionic villus"),
    "extravillous trophoblast": uid("placenta"), "preimplantation embryo": uid("embryo"),
    # heart, joints, fat
    "right atrial appendage": uid("right atrium auricular region"),
    "right atrial appendage biopsies": uid("right atrium auricular region"),
    "synovium": uid("synovium"), "synovial biopsy": uid("synovium"), "synovial tissue": uid("synovium"),
    "bulk synovial tissue": uid("synovium"),
    "visceral adipose tissue": uid("visceral abdominal adipose tissue"),
    # diseases that imply a site (not tumors)
    "ulcerative colitis": uid("large intestine"),
    "crohns disease": uid("intestine"), "crohn's disease": uid("intestine"),
    # ---- added in the second curation pass ----
    # hematopoietic: leukemias and mixed sources
    "acute lymphoblastic leukemia": HEMATO, "acute myeloid leukemia": HEMATO, "leukemia blast cells": HEMATO,
    "bm/pb": HEMATO, "pb/bm": HEMATO, "bonemarrow": uid("bone marrow"),
    "cbnk": uid("umbilical cord blood"), "cord blood pre-cdcs": uid("umbilical cord blood"),
    # skeletal muscle
    "quadriceps muscle": uid("quadriceps femoris"), "musculus vastus lateralis": uid("vastus lateralis"),
    # nervous system: regions written without the organ
    "temporal": uid("temporal lobe"), "frontal": uid("frontal lobe"),
    "lateral temporal lobe": uid("temporal lobe"),
    "cortex ba4": uid("primary motor cortex"), "cortex motor unspecified": uid("primary motor cortex"),
    "ba9 prefrontal cortex": uid("dorsolateral prefrontal cortex"),
    "dorsal lateral prefrontal cortex": uid("dorsolateral prefrontal cortex"),
    "anterior insula": uid("insula"), "orbitofrontal": uid("orbitofrontal cortex"),
    "anterior putamen": uid("putamen"), "posterior putamen": uid("putamen"),
    "anterior caudate": uid("caudate nucleus"), "posterior caudate": uid("caudate nucleus"),
    "cerebellar cortex: anterior and posterior quadrangulate lobules": uid("cerebellar cortex"),
    "lateral amygdala": uid("lateral amygdaloid nucleus"),
    # heart
    "left ventricle apex": uid("heart left ventricle"),
    # airways and nose
    "bronchial brushing": uid("epithelium of bronchus"), "bronchial brushings": uid("epithelium of bronchus"),
    "broncho-alveolar epithelia": uid("respiratory tract epithelium"),
    "human tracheobronchial epithelial cells": uid("tracheobronchial epithelium"),
    "tracheobronchial epithelial cells": uid("tracheobronchial epithelium"),
    "polarized bronchial epithelial cells": uid("epithelium of bronchus"),
    "large airway epithelium": uid("epithelium of bronchus"),
    "large airway basal cells": uid("epithelium of bronchus"),
    "small airway epithelial cells": uid("epithelium of bronchiole"),
    "nasal polyp": uid("nasal cavity mucosa"), "nasal polyps": uid("nasal cavity mucosa"),
    "ascites": uid("ascitic fluid"),
    # reproductive system
    "adnexa": uid("adnexa of uterus"), "ovaries": uid("ovary"), "endometria": uid("endometrium"),
    "cervix": uid("uterine cervix"),
    "uterine fluid derived extracellular vesicles": uid("uterus"),
    # digestive system
    "colorectum": uid("large intestine"), "colonoscopic": uid("colon"),
    "intestinal rectosigmoid": uid("rectosigmoid junction"),
    "islets of langerhans": uid("islet of langerhans"),
    # head and neck
    "larynx and hypopharynx": uid("hypopharynx"),
    # metastases whose site is stated
    "gastric cancer, liver": uid("liver"), "brain metastases,primary breast cancer": uid("brain"),
    # blood, marrow, CSF
    "pb": BLOOD, "bmmc": uid("bone marrow"), "bmmcs": uid("bone marrow"), "bone marrow mononuclear cells": uid("bone marrow"),
    "csf leukocytes": uid("cerebrospinal fluid"), "csf cells": uid("cerebrospinal fluid"),
    # brain regions and their abbreviations
    "dlpfc": uid("dorsolateral prefrontal cortex"), "cortex ba9": uid("dorsolateral prefrontal cortex"),
    "ofc": uid("orbitofrontal cortex"), "prefrontal cortical": uid("prefrontal cortex"),
    "cingulate": uid("cingulate cortex"), "anterior cingulate": uid("cingulate cortex"),
    "basalganglia": uid("basal ganglion"), "purkinje cells": uid("cerebellum"),
    "purkinje primary cell": uid("cerebellum"),
    # heart, vessels, muscle, joints, bone, fat
    "left ventricular": uid("heart left ventricle"), "apical left ventricle": uid("heart left ventricle"),
    "carotid endarterectomy plaque sections": uid("carotid artery segment"),
    "carotid plaque": uid("carotid artery segment"), "carotid atherosclerotic plaque": uid("carotid artery segment"),
    "rectus femoris muscle": uid("rectus femoris"),
    "airway smooth muscle": uid("respiratory system smooth muscle"),
    "airway smooth muscle primary cells": uid("respiratory system smooth muscle"),
    "shoulder capsule": uid("articular capsule of glenohumeral joint"),
    "tenosynovium": uid("tendon sheath"), "tenosynovium sourced during cts decompression surgery": uid("tendon sheath"),
    "cortical fibular bone": uid("fibula"),
    "abdominal fat": uid("adipose tissue of abdominal region"), "scbat": uid("brown adipose tissue"),
    "omental": uid("omentum"), "uterine myometrium": uid("myometrium"),
    # airways, nose, lung
    "bronchial epithelial cell": uid("epithelium of bronchus"), "tracheobronchial": uid("tracheobronchial epithelium"),
    "airway basal cells": uid("respiratory tract epithelium"),
    "alveolar macrophage": uid("lung"), "bronchoalveolar lavage fluid": uid("lung"), "balf": uid("lung"),
    "nasal polyp inferior turbinate": uid("nasal cavity mucosa"), "inferior turbinate": uid("inferior nasal concha"),
    "sputum airway cells": uid("sputum"),
    # eye, kidney, gut, mouth
    "rpe": uid("pigmented layer of retina"), "retinal pigment epithelial cell": uid("pigmented layer of retina"),
    "human iris pigment epithelial cell": uid("iris"), "glomeruli": uid("renal glomerulus"),
    "proximal colon": uid("right colon"), "leftcolon": uid("left colon"), "rightcolon": uid("right colon"),
    "hemorrhoid tissue/anal canal": uid("anal canal"), "masticatory mucosa": uid("mouth mucosa"),
    # reproductive system, placenta, embryo
    "placental villous": uid("chorionic villus"), "placental villi": uid("chorionic villus"),
    "chorioamniotic membranes": uid("fetal membrane"), "chorioamniotic membrane": uid("fetal membrane"),
    "8-cell blastomere": uid("embryo"), "blastomere": uid("embryo"), "blastomeres": uid("embryo"),
    "human oocytes and somatic cells": uid("ovary"), "granulosa cells": uid("ovary"),
    # skin
    "lcs": SKIN,                                                        # Langerhans cells (CD207+) from skin
    # blood fractions, fluids
    "exosome from plasma": uid("blood plasma"), "menstrual blood": uid("menstrual fluid"),
    "urinary extracellular vesicles": uid("urine"), "urinary cells": uid("urine"), "urine": uid("urine"),
    "csf-derived leukocytes": uid("cerebrospinal fluid"), "splenocytes": uid("spleen"),
    # brain
    "cingulate gyrus 25": uid("cingulate gyrus"), "medial temporal gyrus": uid("middle temporal gyrus"),
    "middle temporal gyrus": uid("middle temporal gyrus"), "dentate granule cells": uid("dentate gyrus of hippocampal formation"),
    "ruptured intracranial aneurysm": uid("cerebral artery"), "unruptured intracranial aneurysm": uid("cerebral artery"),
    "intracranial aneurysm": uid("cerebral artery"),
    # heart and vessels
    "left atria": uid("left cardiac atrium"), "right atria": uid("right cardiac atrium"),
    "tricuspid aortic valve leaflets": uid("aortic valve leaflet"), "aortic valve leaflets": uid("aortic valve leaflet"),
    "coronary plaque": uid("coronary artery"),
    # muscle, joints, teeth, fat
    "bilateral paraspinal muscle": uid("skeletal muscle tissue"), "paraspinal muscle": uid("skeletal muscle tissue"),
    "acl tears remnants": uid("anterior cruciate ligament of knee joint"), "acl": uid("anterior cruciate ligament of knee joint"),
    "adult tooth premolar periodontal ligament": uid("periodontal ligament"), "adult tooth pdl": uid("periodontal ligament"),
    "pdl": uid("periodontal ligament"), "subcutaneous gluteofemoral fat": uid("subcutaneous adipose tissue"),
    # airways, nose
    "bronchial aecs": uid("epithelium of bronchus"), "endobronchial": uid("bronchus"),
    "bal macrophages": uid("lung"), "nhne": uid("nasal cavity epithelium"),
    # gut, kidney, mouth
    "sigmoid": uid("sigmoid colon"), "proximal jejunum": uid("jejunum"), "smallbowel": uid("small intestine"),
    "small bowel": uid("small intestine"), "gi": uid("digestive tract"), "ibd": uid("intestine"),
    "iec": uid("intestinal epithelium"), "iecs": uid("intestinal epithelium"),
    "intestinal epithelial cells": uid("intestinal epithelium"), "pancreatic acinar cells": uid("pancreatic acinus"),
    "tubuli": uid("renal tubule"), "gingivo-buccal": uid("mouth mucosa"),
    # eye, embryo
    "rpe/choroid": uid("optic choroid"), "cd31 choroid": uid("optic choroid"),
    "embryo head": uid("embryo"), "embryo limb": uid("embryo"), "single-cell embryo": uid("embryo"),
    "blastocyst fraction": uid("blastocyst"),
    "human islets": uid("islet of langerhans"), "human islets from cadaveric organ donors": uid("islet of langerhans"),
    "cadaveric pancreatic islet": uid("islet of langerhans"), "donor-derived human pancreatic islets": uid("islet of langerhans"),
    # other
    "lung lymph node": uid("lymph node"), "human healthy control plasma": uid("blood plasma"),
}

# tumor names that imply an organ (tagged +tumor); also matched as single words inside longer strings
TUMOR_ORGAN = {
    "glioblastoma": uid("brain"), "gbm": uid("brain"), "glioma": uid("brain"), "astrocytoma": uid("brain"),
    "medulloblastoma": uid("cerebellum"),
    "pdac": uid("pancreas"),
    "hcc": uid("liver"), "hepatoblastoma": uid("liver"), "hepatocarcinoma": uid("liver"),
    "nsclc": uid("lung"), "luad": uid("lung"), "lusc": uid("lung"),
    "ccrcc": uid("kidney"), "rcc": uid("kidney"),
    "nmibc": uid("urinary bladder"), "mibc": uid("urinary bladder"),
    "npc tumor": uid("nasopharynx"),                                  # nasopharyngeal carcinoma ("npc" alone = neural progenitors)
    "primarypdac": uid("pancreas"),
    "aml": HEMATO, "cll": BLOOD,
    # ---- added in the second curation pass ----
    "escc": uid("esophagus"), "eac": uid("esophagus"), "coad": uid("colon"),
    "mpm": uid("pleura"), "mesothelioma": uid("pleura"), "pleural mesothelioma": uid("pleura"),
    "retinoblastoma": uid("retina"),
    "osteosarcoma": uid("bone tissue"), "multiple myeloma": uid("bone marrow"),
    "fibroid": uid("myometrium"), "fibroids": uid("myometrium"), "uterine fibroid": uid("myometrium"),
    "leiomyoma": uid("myometrium"), "uterine leiomyoma": uid("myometrium"),
    "upper tract urothelial carcinoma": uid("upper urinary tract"),
    "upper tract urothelial cell carcinoma": uid("upper urinary tract"),
    "utuc": uid("upper urinary tract"),
    "primary melanoma": SKIN, "melanoma in situ": SKIN, "cutaneous melanoma": SKIN,
    "spitz tumor": SKIN, "spitz nevus": SKIN, "nevus": SKIN, "nevi": SKIN,
    "tubular adenoma": uid("colon"), "colorectal adenoma": uid("large intestine"),
    "ctcl tumor": SKIN, "idc": uid("breast"), "dcis": uid("breast"),
    "hgsoc": uid("ovary"), "hgsc": uid("ovary"), "granulosa cell tumor": uid("ovary"),
    "basal cell carcinoma": SKIN, "bcc": SKIN, "cscc": SKIN, "mpnst": uid("nerve"),
    "basal cell carcinoma tumor": SKIN, "spitzoid melanoma": SKIN,
    "schwannoma": uid("nerve"), "vestibular schwannoma": uid("nerve"),
    "pituitary adenoma": uid("pituitary gland"), "pituitary adenomas": uid("pituitary gland"),
    "seminoma": uid("testis"), "ocscc": uid("oral cavity"), "oscc": uid("oral cavity"),
    "sessile serrated adenoma": uid("colon"), "sessile serrated adenoma/polyp": uid("colon"),                                                # cutaneous T-cell lymphoma
    "hepatocellular carcinoma": uid("liver"), "cholangiocarcinoma": uid("bile duct"),
    "ependymoma": uid("central nervous system"), "spinal ependymoma": uid("spinal cord"),
    "pfa ependymoma": uid("hindbrain"), "dipg": uid("pons"), "diffuse intrinsic pontine glioma": uid("pons"),
}

# strings that never identify a tissue on their own: resolve later from other fields
NOT_A_TISSUE = {
    # tumors without an organ
    "tumor", "tumour", "tumors", "cancer", "primary tumor", "resected tumors", "tumor tissue",
    "baseline tumor tissue", "tumor cells", "tumor containing tissue", "lesion", "crpc metastasis",
    "human melanoma cells", "melanoma cells", "malignant ascites of agc", "dlbcl", "gep-net",
    "hnsc single-cell", "head and neck squamous cell carinoma (hnscc)", "head and squamous cell carcinoma",
    "single acute myeloid leukemia intitiating cell", "tumor infiltrating lymphocytes (tils)", "candidate ctc",
    "head and neck squamous cell carcinoma", "head & neck", "lscc tissue", "leukemia cells", "primary tumor ffpe",
    "cuimc meningioma patient tumor sequencing project",
    # cell types without a site
    "macrophages", "macrophage", "primary macrophages", "microglia", "endocrine cells", "endothelial cells",
    "chondrocytes", "cardiomyocytes", "astrocytes", "keratinocytes", "neuron", "neurons", "neuronal progenitors",
    "primordial germ cells", "oocyte", "ejaculated spermatozoa", "stem cells", "stem cell", "epcam+ cells",
    "cpm-high cells", "cell", "cells", "interneurons", "immune cell", "erythroid cells", "hepatocytes",
    "human retinal pigment epithelial cells", "rna from sorted cells",
    # generic words and leftovers
    "normal", "control", "sample", "biopsy", "biopsies", "periphery", "cell line", "human post mortem tissue",
    "transcriptome of single cells", "cdna", "neuronal epithelium", "inlet cells", "dahoun et al 2008",
    "hcs", "wbs", "ensc", "ls", "hn137p", "phlo", "hb", "nac", "tt", "nl",
    "adjacent normal", "adjacent normal mucosa tissue", "surgical specimen", "pre-treatment endoscopic biopsies",
    "knee/hip",
    # ---- added in the second curation pass ----
    # tumors / lesions without an organ
    "melanoma metastases", "melanoma tumor", "melanoma", "metastatic tumor", "metastatic", "metastasis",
    "metastases", "non-tumor", "nontumor", "squamous cell carcinoma", "hnscc", "sarcoma", "sarcoma tumor",
    "soft tissue", "til", "tils", "cancer stem cells", "disseminated tumor cells", "dtcs", "ctcs",
    "adjacent", "adjacent normal tissue", "normal adjacent", "paracancerous normal", "paracancerous",
    "adjacent non-tumor", "tumor adjacent", "peritumoral", "follicular lymphoma", "lymphoma",
    "of pre-immunotherapy tumor", "single cell suspension of tumor", "patient tumor material",
    # cell types without a site
    "sperm", "hepatocyte", "keratinocyte", "astrocyte", "adipocyte", "adipocytes", "epithelial cells",
    "somatic cells", "natural killer cells", "t follicular helper cells", "cortical interneurons",
    "neural progenitors", "pyramidal neurons", "germ cells", "oocytes", "lymphoblast", "tet", "sorted mdc",
    "sorted nk", "memory cd4+", "modcs",
    # generic words, sample descriptions and unresolvable codes
    "formalin fixed paraffin embedded", "formalin-fixed paraffin-embedded", "native", "tissue mixture",
    "patient sample", "primary patient sample", "patient", "healthy adult woman", "healthy adult man",
    "total rna", "extracellular vesicles", "exosomes", "r", "l", "right", "left", "tp", "nms", "uipsm",
    "npc", "2d epcam+ scrna-seq",
    "polyp", "polyps", "adenoma", "uninvolved", "nonlesion", "non-lesional", "lesional", "dissociated dissected ventricle",
    "striatal gabaergic neurons", "ctfhtet", "cd138+ sorted cells", "healthy adult", "covid-19 positive samples",
    "ex vivo derived cells", "vascular endothelial cells", "melanocytes", "forebrain-type cortical neurons",
    "dm", "gm", "infusion products", "4su-rna", "micro-tissue of primary tumor", "al amyloidosis", "ltt", "proximal",
    "distal", "brisk til", "lyphoma specimen", "lymphoma specimen", "mantle cell lymphoma", "head & neck squamous cancer tumor",
    "circulating tumor cells", "circulating tumor cells sorted from peripheral blood", "villi",
    "osteoblast", "osteoblasts", "foam cell", "foam cells", "leiomyosarcoma", "adjacent non-tumour", "non-tumour",
    "frozen or formalin-fixed tumor", "inflammatory regression", "human fetus", "derma", "mucosal", "pericyte",
    "pericytes", "healthy donor", "microvascular endothelial cells", "endothelial cell", "endothelial", "ffpe",
    "neural cells", "head and neck cells", "teratoma", "human neuroblastoma cells", "uninfected", "wm", "m2",
    "from primary head and neck tumor", "placebo arm", "scleroderma patient", "patient tumor explant",
    "er+/pr+ human tumor explants", "adenocarcinoma", "primary patient derived tumor model", "recurrent tumors",
    "pure non-solid ct density tumor", "dopaminergic neurons", "human cortical excitatory neurons",
    "primary tumor tissue of hnscc", "cd138+ cells", "synovial sarcoma", "synovial sarcoma cells",
    # UBERON labels too generic to be useful; a variant equal to one of these is never looked up
    "tissue", "tissues", "organ", "epithelium", "epithelial", "cortex", "duct", "region", "normal tissue", "nodule",
}

CONTROL_RNA = re.compile(r"reference rna|\buhrr?\b|\bhbrr\b|\bmaqc\b|\bseqc\b|\berccs?\b|universal human reference|"
                         r"^mix of a and b\b|"
                         r"\bhuman reference rna\b")
XENOGRAFT = re.compile(r"\bpdx\b|\bpdx\d{2,}|\bpdx[2-9]\b|\bpdox\b|xenogra(ft|ph)|graft in (rat|mouse|mice)|engrafted|"
                       r"\bnsg\b|humani[sz]ed mice?|mouse stroma|"
                       r"(grown|implanted|injected) in (an? )?(immune[- ]?defici\w+|nude|scid) (mouse|mice)")
CELL_LINE_NAMES = re.compile(
    r"\b(u87|a375|jurkat|huvecs?|mcf-?7|hela|hek-?293t?|293t?|k562|hepg2|a549|lncap|thp-?1|sh-?sy5y|hct-?116|caco-?2|"
    r"mda-?mb-?\d+|ht-?29|panc-?1|u2os|imr-?90|bj|rpe-?1|h295r|wm\d+\w*|coriell|atcc|lcls?|lymphoblastoid|"
    # ---- added in the second curation pass ----
    r"mv-?4;?-?11|molm-?13|kasumi-?1|hl-?60|nb4|oci-?(aml|ly)-?\d+|raji|ramos|daudi|nalm-?6|jk-?1|"
    r"calu-?3|h1299|h460|pc-?9|hcc827|beas-?2b|16hbe\w*|sk-?br-?3|skbr-?3|t-?47-?d|bt-?20|bt-?474|bt-?549|zr-?75-?1|"
    r"sum-?\d+\w*|cal-?51|hs578t|hmle|mcf-?10a|sk-?hep-?1|huh-?7|hep-?3b|ags|mkn-?\d+|snu-?\d+|"
    r"sw-?480|sw-?620|lovo|dld-?1|hct-?15|rko|lim1215|pc-?3|du-?145|22rv1|vcap|c4-?2b|"
    r"sk-?ut-?1|syo-?1|g402|a673|tc-?71|rh(18|28|30|36|41)|khos|sjsa-?1?|saos-?2|mg-?63|143b|u-?251|t98g|ln-?229|sk-?n-?(sh|as|be|dz|mc|fi)\w*|be2c|imr-?32|"
    r"ovcar-?\d+|skov-?3|peo-?[14]|caov-?3|heya8|ishikawa|hec-?1-?[ab]|siha|caski|c-?33a|"
    r"786-?o|caki-?\d|ach-?n|hk-?2|hacat|hues\d+|bobsc|gm\d{4,5}|ebv-?b|hn\d{3}\w*|"
    r"nci-?h\d+|hcc-?\d+|mda-?mba?-?\d+|um-?scc-?\d+\w*|scc-?\d+|cal-?27|fadu|detroit ?562|mono-?mac-? ?6|kgn|"
    r"rpe-? ?1|arpe-?19|htert\w*|meg-?01|sk-?mel-?\d+|hek-?293\w*|hela\w*|meso\d+|lx-?2|t-?84|sh-?sy-?5y|"
    r"ntera-?2|sf\d{3,4}|dbtrg\w*|lhcn-?m2|12z|\w+/e6e7|ha1800|telo\w+|hs-?sy-?ii)\b|"
    r"umbilical vein endothelial|\bcell lines?\b|ebv[- ]transformed|gencord|immortali[sz]ed")
IN_VITRO = re.compile(
    r"\bips\b|ipsc|induced pluripotent|embryonic stem|\bescs?\b|\bhescs?\b|\bh?i?pscs?\b|\bh[179]\b|\bhes\b|"
    r"(pluripotent|neural|mesenchymal|embryonic) stem cells?|\bh?mscs?\b|"
    r"(dental pulp|limbal|amniotic fluid|adipose[- ]derived|wharton'?s jelly) (mesenchymal )?stem cells?|"
    r"embryoid|organoid|enteroid|colonoid|assembloid|blastoid|spheroid|organotypic|stem[- ]cell[- ]derived|"
    r"progenitor cell[- ]derived|monocytes?[- ]derived|psc[- ]derived|\bmo-?dcs?\b|\bmdms?\b|\bmdmi\b|"
    r"periosteum-derived|"
    r"derived from h?ipsc|progenitor cells|differentiat|\binduced (neurons?|astrocytes?|\w+ cells?)|-induced|\bicell\b|"
    r"ngn2|arc-type neurons|\bineurons?\b|neuron-like|reprogramm|hepatopac|hif-ts?\b|\bhwjscs?\b|stromal cells?\b|"
    r"tissue culture|\b[23]d (culture|cells?)\b|\bcells? [23]d\b|in vitro|\bivc\b|air[- ]liquid interface|\bcar[- ]?t\b|"
    r"in culture|co-?culture|seeded|decellulari|trophoblast stem|\bkips\b|chemical reset|\bes[- ]derived|\bnpcs\b|"
    r"\bimgls?\b|\bcafs?\b|\bhrecs?\b|\bhlecs?\b|\bhsaecs?\b|"
    r"cell culture|cultured|passage|fibroblasts?\b|\bhffs?\b|myotubes?|myoblasts?|engineered|programmed|"
    r"aortic endothelial|valve endothelial|mucilair|epidermal keratinocytes?|on-a-chip|\w+ chip$|"
    r"\b(nheks?|hmecs?|hcaecs?|hpaecs?|hpmecs?|hdmecs?|hbmecs?|hasmcs?|haoecs?|hpmcs?)\b|"   # primary cultures, not cell lines
    r"^primary .*\b(cells?|hepatocytes|keratinocytes|myotubes)\b|^dcx[+-] d\d+$|"
    r"^(human )?primary( human)? (\w+ )?(hepatocytes?|keratinocytes?|astrocytes?|chondrocytes?|fls|melanocytes?|"
    r"adipocytes?)$")
# sample codes and protocol descriptions that carry no tissue ("bc05 single-cell rna-seq", "mm34em", "patient 2 ...");
# only used once every other step found nothing
SAMPLE_CODE = re.compile(r"^(patient|subject|donor|sample|case|bc|mm|mgh|p)[ -]?\d+\w*$|"
                         r"^([\w-]+ ){0,2}single[- ]cell m?rna-seq( [\w-]+)?$|"
                         r"^p\d+ t\d.*clone|^formalin[- ]fixed paraffin[- ]embedded biopsies or archival|"
                         r"^(tumou?r|normal)-? ?\d+$|\bctcs?\b|circulating tumou?r cells")
HEMATO_RE = re.compile(r"cd34\+|\bcd34(?![-\w])|hematopoietic (stem |progenitor )?(and progenitor )?cells?|\bhspcs?\b|\bhscs?\b|"
                       r"megakaryocyte|erythroid progenitor|myeloid progenitor|hematopoietic (stem|progenitor)s?\b|"
                       r"erythroblasts?")
NASAL_EPI = re.compile(r"nasal[- ](brush|brushing|epitheli|airway|scrap|swab|curett|rna)|brushing of nasal")
NASOPHARYNX = re.compile(r"naso-?pharyn|oro-?naso-?pharyn")
# blood rule: a blood word or blood-cell word, and no other site
BLOOD_WORD = re.compile(r"\b(whole[- ]?)?blood\b|\b(leuk)?apheresis\b")
BLOOD_CELL_WORD = re.compile(
    r"\b(t|b|nk|nkt)[- .]?cells?\b|\btcells?\b|\bbcells?\b|\btregs?\b|\bpbmcs?\b|\bpbls?\b|natural killer|"
    r"\bcd(3|4|8|14|16|19|20|56)[+-]? (t |memory |naive |effector )?(cells?|memory|naive|effector)\b|"   # "cd8 naive"
    r"monocytes?|neutrophils?|thrombocytes?|platelets?|erythrocytes?|granulocytes?|lymphocytes?|leukocytes?|"
    r"eosinophils?|basophils?|plasma ?cells?|plasmablasts?|dendritic cells?|"
    r"\bnaive (cd\d+|b|t)\b|regulatory t|t regulatory|\bth-?(1|2|17)\b|\bmait\b|\bpdcs?\b|"
    r"^(cd\d+[+-]?|ccr\d+[+-]?)+$")                                  # bare marker strings: cd4, cd3+cd4+ccr7+
NON_BLOOD_SITE = re.compile(r"tumou?r|infiltrat|tissue|biops|lavage|synovi|skin|lung|liver|gut|intestin|colon|"
                            r"brain|csf|marrow|tonsil|lymph node|spleen|thymus|decidua|placenta|derived|culture|"
                            r"cord|menstrual")
TUMOR_WORDS = re.compile(r"\b(tumou?rs?|cancers?|carcinomas?|carinomas?|adenocarcinomas?|neoplasms?|"
                         r"metastas[ie]s|metastatic|malignant|primary|resected|baseline)\b")
# words that only say "the normal part" of a tumor sample; dropped together with the tumor words (no +tumor tag)
NORMAL_WORDS = re.compile(r"\b(adjacent|normal|paracancerous|para-?carcinoma|peri-?tumou?ral|non-?tumou?r(al|ous)?|"
                          r"non-?cancerous|non-?malignant|uninvolved|benign|matched|healthy)\b")

# last resort: an organ word anywhere in the string (multi-word names first); ambiguous if two organs appear
ORGAN_WORDS = {w: uid(lbl) for w, lbl in [
    ("bone marrow", "bone marrow"), ("spinal cord", "spinal cord"), ("lymph node", "lymph node"),
    ("adipose", "adipose tissue"), ("small intestine", "small intestine"), ("liver", "liver"), ("lung", "lung"),
    ("kidney", "kidney"), ("heart", "heart"), ("brain", "brain"), ("cerebellum", "cerebellum"), ("colon", "colon"),
    ("stomach", "stomach"), ("pancreas", "pancreas"), ("spleen", "spleen"), ("thymus", "thymus"), ("skin", "skin of body"),
    ("breast", "breast"), ("prostate", "prostate gland"), ("ovary", "ovary"), ("testis", "testis"), ("uterus", "uterus"),
    ("placenta", "placenta"), ("retina", "retina"), ("esophagus", "esophagus"), ("bladder", "urinary bladder"),
    ("thyroid", "thyroid gland"), ("adrenal", "adrenal gland"), ("tonsil", "tonsil"), ("rectum", "rectum"),
    ("ileum", "ileum"), ("duodenum", "duodenum"), ("endometrium", "endometrium"), ("trachea", "trachea"),
    # ---- added in the second curation pass (only organs that rarely co-occur with another organ word;
    # "artery", "bone", "eye", "cervix", "pleura", "omentum" or "umbilical cord" made "pulmonary artery",
    # "prostate bone metastasis" or "spinal cord cervical" ambiguous) ----
    ("peritoneum", "peritoneum"), ("vagina", "vagina"), ("vulva", "vulva"),
    ("fallopian tube", "fallopian tube"), ("gallbladder", "gallbladder"), ("bile duct", "bile duct"),
    ("cecum", "caecum"), ("appendix", "vermiform appendix"), ("tongue", "tongue"),
    ("larynx", "larynx"), ("pharynx", "pharynx"), ("oral cavity", "oral cavity"), ("salivary gland", "salivary gland"),
    ("parotid", "parotid gland"), ("cornea", "cornea"), ("conjunctiva", "conjunctiva"),
    ("hypothalamus", "hypothalamus"), ("skeletal muscle", "skeletal muscle tissue"), ("tendon", "tendon"),
    ("cartilage", "cartilage tissue"), ("urethra", "urethra"), ("penis", "penis"), ("epididymis", "epididymis"),
    ("bronchoalveolar lavage", "lung"), ("hair follicle", "hair follicle"), ("synovium", "synovium"),
    ("myometrium", "myometrium")]}
ORGAN_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, ORGAN_WORDS), key=len, reverse=True)) + r")s?\b")   # "lungs"

# text repair (the text is already lowercased, so broken UTF-8 is fixed by direct replacement)
MOJIBAKE = {"ã¯": "i", "ã©": "e", "ã¨": "e", "ã¶": "o", "ã¼": "u", "ã¤": "a", "ã±": "n", "â€“": "-", "â€™": "'"}
TYPOS = {"tisue": "tissue", "tisssue": "tissue", "periperal": "peripheral", "perpheral": "peripheral",
         "umblical": "umbilical", "oronasopharingeal": "oronasopharyngeal",
         "lymphoycyte": "lymphocyte", "lymphoycytes": "lymphocytes",
         "pacreatic": "pancreatic", "substanstia": "substantia", "medullobalstoma": "medulloblastoma",
         "plalelets": "platelets", "lympohoma": "lymphoma", "ketratinocytes": "keratinocytes",
         "periphheral": "peripheral", "esopheageal": "esophageal", "artial": "atrial", "mesencymal": "mesenchymal",
         "keratinoocyte": "keratinocyte", "keratinoctyes": "keratinocytes", "oseochondroma": "osteochondroma",
         "biospy": "biopsy", "ligment": "ligament", "plancenta": "placenta", "umbillical": "umbilical",
         "hematopoeitic": "hematopoietic", "spleenocytes": "splenocytes", "medil": "medial", "carinoma": "carcinoma",
         "lymphnode": "lymph node", "lymphnodes": "lymph nodes"}
ADJ = {"rectal": "rectum", "ileal": "ileum", "colonic": "colon", "gastric": "stomach", "hepatic": "liver",
       "renal": "kidney", "pulmonary": "lung", "cardiac": "heart", "duodenal": "duodenum", "jejunal": "jejunum",
       "esophageal": "esophagus", "oesophageal": "esophagus", "pancreatic": "pancreas", "splenic": "spleen",
       "thymic": "thymus", "dermal": "dermis", "cutaneous": "skin", "gingival": "gingiva",
       "endometrial": "endometrium", "placental": "placenta", "tonsillar": "tonsil", "bronchial": "bronchus",
       "tracheal": "trachea", "ovarian": "ovary", "testicular": "testis", "prostatic": "prostate",
       "endocervical": "endocervix", "cervical": "cervix", "mammary": "breast",
       "colorectal": "large intestine", "hepatocellular": "liver", "synovial": "synovium",
       # ---- added in the second curation pass ----
       "peritoneal": "peritoneum", "pleural": "pleura", "ectocervical": "ectocervix", "vaginal": "vagina",
       "vulvar": "vulva", "uterine": "uterus", "cecal": "cecum", "biliary": "bile duct",
       "hippocampal": "hippocampus", "hypothalamic": "hypothalamus", "cerebellar": "cerebellum",
       "osseous": "bone", "oral": "oral cavity", "lingual": "tongue", "laryngeal": "larynx",
       "pharyngeal": "pharynx", "salivary": "salivary gland", "ocular": "eye", "corneal": "cornea",
       "conjunctival": "conjunctiva", "ureteral": "ureter", "urethral": "urethra", "penile": "penis",
       "decidual": "decidua", "intestinal": "intestine", "adrenocortical": "adrenal",
       "myometrial": "myometrium"}

In [14]:
# ---------------- 3) matcher ----------------
PREFIX = re.compile(r"^(human|donor \d+|donor|normal|healthy|adult|fetal|foetal|embryonic|primary|fresh|frozen|"
                    r"cryopreserved|total|whole|ffpe|post-?mortem|paired|"
                    r"neonatal|newborn|p(a)?ediatric|infant|juvenile|pregnant|bulk|pooled|homogeni[sz]ed|explanted|"
                    r"laser[- ]captured|lcm[- ]dissected|lcm|isolated|sorted|control)\s+")
SUFFIX = re.compile(r"\s+(tissue|tissues|sample|samples|biopsy|biopsies|specimen|specimens|punch)$")
CELLS  = re.compile(r"\s+(cells?|cell population|"
                    r"((pyramidal|dopamine|dopaminergic|gabaergic|medium spiny) )?neurons?)$")


# Variants of a given string are generated using the following mechanisms:
# 1) Identity: original candidate is always included
# 2) PREFFIX and SUFFIX regexes are applied iteratively up to 3 times
# 3) Tracked adjectives are subsituted to nouns
# 4) Plurals are eliminated
# 5) CELL regexes (cells, cell population, neurons) are removed
# 6) "tissue" and "organ" are appended 
def _variants(s):
    out, seen = [], set()
    def add(x):
        x = x.strip(" ,;-")
        if x and x not in seen:
            seen.add(x); out.append(x)
    add(s); x = s
    for _ in range(3):
        x = SUFFIX.sub("", PREFIX.sub("", x)); add(x)
    for v in list(out):                                   # adjective -> noun ("rectal biopsy" -> "rectum")
        w = v.split(" ", 1)
        if w[0] in ADJ:
            add(SUFFIX.sub("", " ".join([ADJ[w[0]]] + w[1:])))
    for v in list(out):
        if v.endswith("s") and not v.endswith("ss"):
            add(v[:-1])
    for v in list(out):                                   # weaker variants last
        add(CELLS.sub("", v)); add(v + " tissue"); add(v + " organ")
    return out

# Additional variants are generated by removing parenthesis and tumor-related words
def variants(s):
    """Variants of s, then of s without (parentheses) / "week N", then without tumor and "adjacent normal" words."""
    out = _variants(s)
    no_paren = re.sub(r"\s*\(.*?\)", "", s).strip()
    no_paren = re.sub(r"\s+(of\s+)?(gestational\s+)?weeks?\s+\d+", "", no_paren).strip()   # "... of week 24"
    no_tumor = re.sub(r"\s+", " ", NORMAL_WORDS.sub("", TUMOR_WORDS.sub("", no_paren))).strip()   # "adjacent normal" too
    for extra in (no_paren, no_tumor):
        if extra and extra != s:
            out += [v for v in _variants(extra) if v not in out]
    return out

def paren_variants(s):
    """Variants of the text inside parentheses ("primary tumor (colorectum)" -> "colorectum"). They are the weakest
    evidence: used only after the exclusion rules, and never for RELATED synonyms. Short codes like "(nc)" are skipped."""
    out = []
    for inner in re.findall(r"\(([^()]{4,})\)", s):
        out += [v for v in _variants(inner.strip()) if v not in out]
    return out

def fix_text(s):
    for bad, good in MOJIBAKE.items():
        s = s.replace(bad, good)
    s = s.replace("ï", "i").replace("é", "e")
    s = re.sub(r"â(?=\s|,|$)", "", s)                     # a mangled non-breaking space: "umbilicalâ cordâ blood"
    return " ".join(TYPOS.get(w, w) for w in s.split())

def _hit(tid, how):
    return tid, terms[tid]["name"], how

# Function to match a string to an UBERON ID
def match(s):
    s = fix_text(s)
    vs = variants(s)
    ps = [v for v in paren_variants(s) if v not in vs]
    tumor = "+tumor" if TUMOR_WORDS.search(s) else ""
    for v in vs:                                          # explicit decisions first, on every variant
        if v in MANUAL:
            return _hit(MANUAL[v], "manual")
    for v in vs:
        if v in TUMOR_ORGAN:
            return _hit(TUMOR_ORGAN[v], "tumor name+tumor")
    if s in NOT_A_TISSUE:
        return None, None, "not a tissue"
    if CONTROL_RNA.search(s):
        return None, None, "control rna"
    if XENOGRAFT.search(s):
        return None, None, "xenograft"
    if CELL_LINE_NAMES.search(s):
        return None, None, "cell line name"
    if IN_VITRO.search(s):
        return None, None, "in vitro"
    if HEMATO_RE.search(s):
        return _hit(HEMATO, "hematopoietic rule")
    if NASAL_EPI.search(s):
        return _hit(uid("nasal cavity epithelium"), "nasal rule")
    if NASOPHARYNX.search(s):
        return _hit(uid("nasopharynx"), "nasal rule")
    if (BLOOD_WORD.search(s) or BLOOD_CELL_WORD.search(s)) and not NON_BLOOD_SITE.search(s):
        return _hit(BLOOD, "blood rule")
    for w in re.findall(r"[a-z0-9]+", s):               # tumor abbreviations inside longer strings
        if w in TUMOR_ORGAN:
            return _hit(TUMOR_ORGAN[w], "tumor name+tumor")
    for v in ps:                                          # explicit decisions on the text inside parentheses
        if v in MANUAL:
            return _hit(MANUAL[v], "manual")
        if v in TUMOR_ORGAN:
            return _hit(TUMOR_ORGAN[v], "tumor name+tumor")
    for kinds in (("label", "exact"), ("related",)):    # strong evidence on all variants first
        for v in (vs + ps if kinds[0] == "label" else vs):
            if v in NOT_A_TISSUE:                         # "normal tissue" -> "tissue" is a label, but says nothing
                continue
            for kind in kinds:
                if kind == "related" and v != s and v.endswith((" tissue", " organ")):
                    continue                              # "frontal" + " organ" is a RELATED synonym of the pineal body
                ids = lookup[kind].get(v)
                if ids:
                    if len(ids) > 1:
                        return None, None, f"ambiguous {kind}"
                    return _hit(next(iter(ids)), kind + ("" if v == s else "+variant") + tumor)
    organs = {ORGAN_WORDS[m] for m in ORGAN_RE.findall(" ".join(vs[:1] + [ADJ.get(w, w) for w in s.split()]))}
    if len(organs) == 1:
        return _hit(organs.pop(), "organ word" + tumor)
    if len(organs) > 1:
        return None, None, "ambiguous organ words"
    if SAMPLE_CODE.search(s) or any(v in NOT_A_TISSUE for v in vs):   # generic once stripped: "ffpe tumor" -> "tumor"
        return None, None, "not a tissue"
    return None, None, "no match"

# ---------------- 4) match every string in the combined queue ----------------
cur = queues["combined"].copy()
cur[["uberon_id", "uberon_label", "match"]] = pd.DataFrame(cur["string"].map(match).tolist(), index=cur.index)
cur["category"] = np.select(
    [(cur["frac_cell_line"] >= 0.5) | (cur["match"] == "cell line name"),
     cur["match"] == "in vitro",
     cur["match"] == "xenograft",
     cur["match"] == "control rna",
     cur["match"] == "not a tissue",
     cur["uberon_id"].notna()],
    ["cell_line", "in_vitro", "xenograft", "control_rna", "needs_context", "tissue"], default="unmatched")
cur["tumor"] = cur["match"].str.endswith("+tumor")
cur["needs_review"] = (cur["match"].str.startswith(("related", "ambiguous"))
                       | (cur["category"] == "unmatched"))

# ---------------- 5) summary ----------------
n_all = len(samples)
by_cat = cur.groupby("category")["n_samples"].sum().sort_values(ascending=False)
print("samples by category (all strings):")
print((by_cat / n_all).map("{:.1%}".format).to_string())
print("\nmatch type among matched strings (samples):")
print((cur[cur["uberon_id"].notna()].groupby("match")["n_samples"].sum() / n_all).map("{:.1%}".format).to_string())

top = cur.head(N_REVIEW)
print(f"\ntop {N_REVIEW} strings: {top['uberon_id'].notna().mean():.1%} auto-mapped, "
      f"{top['needs_review'].sum():,} need review, covering {top['n_samples'].sum() / n_all:.1%} of samples")

print("\nmost frequent strings still needing a decision (needs_context excluded):")
print(top.loc[top["needs_review"],
              ["field", "string", "n_samples", "n_series", "frac_cell_line", "match"]].head(40).to_string())

# ---------------- 6) attach the automatic labels to the samples ----------------
key = pd.MultiIndex.from_arrays([
    np.where(samples["tissue_value"].notna(), "tissue", "source_name"),
    samples["tissue_value"].fillna(samples["source_name"])])
lab = cur.set_index(["field", "string"])
samples["uberon_auto"]   = lab["uberon_id"].reindex(key).to_numpy()
samples["uberon_match"]  = lab["match"].reindex(key).to_numpy()
samples["category_auto"] = lab["category"].reindex(key).to_numpy()
samples["tumor_auto"]    = lab["tumor"].reindex(key, fill_value=False).to_numpy()
print(f"\nsamples with an automatic UBERON id: {samples['uberon_auto'].notna().mean():.1%}")

samples by category (all strings):
category
tissue           47.2%
cell_line        32.9%
in_vitro          9.0%
needs_context     6.2%
unmatched         3.5%
xenograft         0.8%
control_rna       0.3%

match type among matched strings (samples):
match
blood rule                5.7%
exact                     2.9%
exact+variant             1.9%
exact+variant+tumor       0.5%
hematopoietic rule        0.7%
label                    10.2%
label+tumor               0.0%
label+variant             4.7%
label+variant+tumor       2.2%
manual                   17.2%
nasal rule                0.6%
organ word                2.2%
organ word+tumor          1.0%
related                   0.0%
related+variant           0.0%
related+variant+tumor     0.0%
tumor name+tumor          1.5%

top 2000 strings: 47.1% auto-mapped, 4 need review, covering 72.5% of samples

most frequent strings still needing a decision (needs_context excluded):
            field                                        string 

### 3) Review mapping

In [15]:
samples[samples["uberon_auto"].notna()].iloc[1,:]

series_id                                          GSE40819,GSE40820
source_name                                                 placenta
tissue_key                                                       NaN
tissue_value                                                     NaN
cell_type_value                                                  NaN
cell_line_value                                                  NaN
tissue_fallback                                                  NaN
title                                                      mix3-rep1
characteristics    spike-in hsa-mir-147: 10,spike-in hsa-mir-211:...
has_cell_line                                                  False
uberon_auto                                           UBERON:0001987
uberon_match                                                   label
category_auto                                                 tissue
tumor_auto                                                     False
Name: GSM1002542, dtype: object

In [16]:
review = (samples[samples["uberon_auto"].notna()]
          .groupby("uberon_match", group_keys=False)
          .apply(lambda d: d.sample(min(len(d), 20), random_state=0)))
review["uberon_label"] = review["uberon_auto"].map(lambda t: terms[t]["name"])
review[["uberon_match", "uberon_label", "tissue_value", "source_name", "title"]]
# review

/scratch/slurm-job.9971094/ipykernel_200341/3201527996.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: d.sample(min(len(d), 20), random_state=0)))


,uberon_match,uberon_label,tissue_value,source_name,title
sample_id,,,,,
GSM5565799,blood rule,blood,NaN,blood platelets vumc-pdac-115-2015098-tr2661,vumc-pdac-115-2015098-tr2661
GSM1962511,blood rule,blood,NaN,in.vivo.single.tcell,tcell person1 p1299 1475
GSM3399461,blood rule,blood,NaN,effector memory t cells,l41 v9 cd4 em
GSM4082149,blood rule,blood,NaN,cem clonal t cells,sample 4 cem.chsf1 water 18h
GSM5541700,blood rule,blood,NaN,islet-reactive cd4+ sorted t-cell,21180 h9 new onset t1d 21180 lib33748 ccuwnanx...
...,...,...,...,...,...
GSM5556908,tumor name+tumor,liver,hepatocellular carcinoma,human tumor tissue,hccb36t
GSM3416501,tumor name+tumor,pancreas,NaN,pdac,cumc s108
GSM2735251,tumor name+tumor,nasopharynx,NaN,npc tumor,patient 55


### 4) Map remaining with low confidence

1) Fuzzy mapping to the ontology
2) Use remaining GEO fields
3) Expression classifier
4) Text2term fuzzy mapping

In [23]:
cur

,field,string,n_samples,n_series,frac_cell_line,cum_frac_samples,uberon_id,uberon_label,match,category,tumor,needs_review
0,tissue,blood,22841,482,0.09,0.031749,UBERON:0000178,blood,manual,tissue,False,False
1,tissue,whole blood,15763,172,0.02,0.053659,UBERON:0000178,blood,manual,tissue,False,False
2,tissue,skin,9438,260,0.23,0.066778,UBERON:0000014,zone of skin,exact,tissue,False,False
3,source_name,lcl-derived ipsc,8682,3,0.00,0.078846,None,None,cell line name,cell_line,False,False
4,tissue,peripheral blood,8507,374,0.08,0.090670,UBERON:0000178,blood,manual,tissue,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
46073,source_name,hg02666 gard,1,1,1.00,0.999558,None,None,no match,cell_line,False,False
46074,source_name,hg02622 tunic,1,1,1.00,0.999559,None,None,no match,cell_line,False,False
46075,source_name,hg02625 baff,1,1,1.00,0.999561,None,None,no match,cell_line,False,False
46076,source_name,plasmablast v2d9,1,1,0.00,0.999562,UBERON:0000178,blood,blood rule,tissue,False,False


## UBERON IDs --> CellxGene Categories

### Map to CellxGene Categories

In [17]:
from cellxgene_ontology_guide.ontology_parser import OntologyParser
from cellxgene_ontology_guide.curated_ontology_term_lists import get_curated_ontology_term_list
from cellxgene_ontology_guide.entities import CuratedOntologyTermList

parser = OntologyParser()                       # or OntologyParser(schema_version="v<X.Y.Z>") to match your Census release
TG = get_curated_ontology_term_list(CuratedOntologyTermList.TISSUE_GENERAL)

def to_tissue_general(tid):
    try:
        hits = parser.get_high_level_terms(tid, TG)
        return hits[0] if hits else None
    except (KeyError, ValueError, IndexError):  # id unknown to CELLxGENE's UBERON version
        return None

In [18]:
tg = {t: to_tissue_general(t) for t in samples["uberon_auto"].dropna().unique()}
samples["tissue_general_id"] = samples["uberon_auto"].map(tg)
samples["tissue_general"] = samples["tissue_general_id"].map(lambda t: parser.get_term_label(t) if isinstance(t, str) else None)

unmapped = [t for t, v in tg.items() if v is None]
print(f"{len(tg)} UBERON ids -> {len(set(tg.values()) - {None})} tissue_general categories; {len(unmapped)} ids without one: "
      + ", ".join(f"{t} ({terms[t]['name']})" for t in unmapped[:10]))
print(samples.loc[samples["category_auto"] == "tissue", "tissue_general"].value_counts(dropna=False).head(40).to_string())

print(f"all samples:      {samples['tissue_general_id'].notna().mean():.1%}")
print(f"tissue category:  {samples.loc[samples['category_auto'] == 'tissue', 'tissue_general_id'].notna().mean():.1%}")
print(samples.groupby("category_auto", dropna=False)["tissue_general_id"].apply(lambda s: f"{s.notna().mean():.1%}"))

597 UBERON ids -> 61 tissue_general categories; 59 ids without one: UBERON:0000106 (zygote stage), UBERON:0000173 (amniotic fluid), UBERON:0007376 (outer epithelium), UBERON:0001090 (synovial fluid), UBERON:0004345 (trophectoderm), UBERON:0003728 (mediastinum), UBERON:0002331 (umbilical cord), UBERON:0000305 (amnion), UBERON:0001464 (hip), UBERON:0000403 (scalp)
tissue_general
blood                     135936
skin of body               19462
brain                      18263
pancreas                   17171
bone marrow                15071
liver                      13718
lung                       11512
breast                      7111
hematopoietic system        7024
colon                       5831
fallopian tube              5584
adipose tissue              5276
musculature                 5091
uterus                      4955
heart                       4828
embryo                      4775
nose                        4733
kidney                      4616
large intestine           

## Map back to source H5AD

In [43]:
path = Path("/cluster/work/boeva/rquiles/unifiedFM_experiments/embeddings/pca")
h5ad_file = path / "bulk_test.h5ad"

adata = sc.read_h5ad(h5ad_file, backed="r")

In [41]:
adata.obs.head()

,modality,cell_line,paired,dataset,bulk_source,bulk_dataset,characteristics,sample,series_id,source,organism,tissue_general,orig_obs_name,source_file_merge
37,bulk,COLO792,36,Zhu,0,0,0,0,0,0,0,0,37,bulk_paired
40-1,bulk,JHH6,91,Zhu,0,0,0,0,0,0,0,0,40-1,bulk_paired
6-2,bulk,LOVO,107,Zhu,0,0,0,0,0,0,0,0,6-2,bulk_paired
22-3,bulk,RERFLCAD1,169,Zhu,0,0,0,0,0,0,0,0,22-3,bulk_paired
33-3,bulk,SCC90,180,Zhu,0,0,0,0,0,0,0,0,33-3,bulk_paired


In [44]:
# make sure index types match (both should be str)
adata.obs.index = adata.obs.index.astype(str)
samples.index = samples.index.astype(str)

# aligned on sample_id; samples missing from `samples` become NaN
adata.obs["tissue_general"] = samples["tissue_general"].reindex(adata.obs.index)

print(adata.obs["tissue_general"].isna().mean())   # fraction not matched

0.64995


In [46]:
def add_tissue_general(h5ad_path, samples, column="tissue_general",
                       min_overlap=0.5, overwrite=True):
    """Add `column` from `samples` (indexed by sample_id) to the .obs of an h5ad,
    matching on the index, report the mapped fraction and overwrite the file.

    Flags:
      - duplicated sample_ids in `samples` (raises: alignment would be ambiguous)
      - duplicated sample_ids in adata.obs (warns)
      - obs ids not found in `samples` (reported, left as NaN)
      - obs ids found but with an empty `column` in `samples` (reported)
      - overlap below `min_overlap` (raises and leaves the file untouched,
        which usually means an id format mismatch)
    Returns the updated obs DataFrame.
    """
    h5ad_path = Path(h5ad_path)

    # Read fully into memory: a file opened backed cannot safely be rewritten in place.
    adata = sc.read_h5ad(h5ad_path)
    n = adata.n_obs

    obs_ids = adata.obs.index.astype(str)
    src = samples[column].copy()
    src.index = samples.index.astype(str)

    if src.index.has_duplicates:
        raise ValueError(f"`samples` has {src.index.duplicated().sum():,} duplicated sample_ids")
    if obs_ids.has_duplicates:
        warnings.warn(f"adata.obs has {obs_ids.duplicated().sum():,} duplicated ids")

    in_samples = obs_ids.isin(src.index)
    overlap = in_samples.mean()
    new = src.reindex(obs_ids)
    new.index = adata.obs.index
    mapped = new.notna()

    print(f"{h5ad_path.name}: {n:,} observations")
    print(f"  ids found in samples : {in_samples.sum():,} ({overlap:.1%})")
    print(f"  ids not in samples   : {(~in_samples).sum():,}")
    print(f"  found but empty {column}: {(in_samples & ~mapped.to_numpy()).sum():,}")
    print(f"  mapped fraction      : {mapped.mean():.1%}")
    if (~in_samples).any():
        print(f"  example unmatched ids: {list(obs_ids[~in_samples][:5])}")

    if overlap < min_overlap:
        raise ValueError(
            f"Only {overlap:.1%} of obs ids match `samples` (< {min_overlap:.0%}); "
            "check id format (prefix, case, whitespace). File left untouched."
        )

    if column in adata.obs.columns:
        if not overwrite:
            raise ValueError(f"'{column}' already in obs and overwrite=False")
        print(f"  note: existing '{column}' column replaced")

    adata.obs[column] = new.astype("category")

    # Write to a temp file, then replace atomically so a failed write cannot corrupt the original.
    tmp = h5ad_path.with_name(h5ad_path.name + ".tmp")
    adata.write_h5ad(tmp)
    os.replace(tmp, h5ad_path)
    print(f"  overwrote {h5ad_path}")
    return adata.obs

In [48]:
add_tissue_general(h5ad_file, samples, overwrite=True)

bulk_test.h5ad: 20,000 observations
  ids found in samples : 14,390 (72.0%)
  ids not in samples   : 5,610
  found but empty tissue_general: 7,389
  mapped fraction      : 35.0%
  example unmatched ids: ['37', '40-1', '6-2', '22-3', '33-3']
  note: existing 'tissue_general' column replaced
  overwrote /cluster/work/boeva/rquiles/unifiedFM_experiments/embeddings/pca/bulk_test.h5ad


,modality,cell_line,paired,dataset,bulk_source,bulk_dataset,characteristics,sample,series_id,source,organism,tissue_general,orig_obs_name,source_file_merge
37,bulk,COLO792,36,Zhu,0,0,0,0,0,0,0,NaN,37,bulk_paired
40-1,bulk,JHH6,91,Zhu,0,0,0,0,0,0,0,NaN,40-1,bulk_paired
6-2,bulk,LOVO,107,Zhu,0,0,0,0,0,0,0,NaN,6-2,bulk_paired
22-3,bulk,RERFLCAD1,169,Zhu,0,0,0,0,0,0,0,NaN,22-3,bulk_paired
33-3,bulk,SCC90,180,Zhu,0,0,0,0,0,0,0,NaN,33-3,bulk_paired
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM9469718,bulk,0,0,ARCHS4,ARCHS4,pretraining_bulk_RAW,"tissue: small intestine organoid,cell type: me...",GSM9469718,GSE317332,"Human iPSC (Control 1, clone 1.5; female, 42 y...",Homo sapiens,NaN,GSM9469718,bulk_pretraining
GSM9473061,bulk,0,0,ARCHS4,ARCHS4,pretraining_bulk_RAW,"cell line: BXPC3,cell type: pancreatic cancer ...",GSM9473061,GSE317494,BXPC3,Homo sapiens,NaN,GSM9473061,bulk_pretraining
GSM9476780,bulk,0,0,ARCHS4,ARCHS4,pretraining_bulk_RAW,"tissue: Gonad,cell line: 9 months,cell type: 4...",GSM9476780,GSE317696,Gonad,Homo sapiens,NaN,GSM9476780,bulk_pretraining
GSM986133,bulk,0,0,ARCHS4,ARCHS4,pretraining_bulk_RAW,"cell line: HEK293,sirna: none,fragmentation: a...",GSM986133,GSE40137,HEK293,Homo sapiens,NaN,GSM986133,bulk_pretraining


## Plot PCA with new labels

In [19]:
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

BULK_DIR = Path("/cluster/work/boeva/rquiles/data/ARCHS4-data/data/processed/archs4")
EXPRESSION_H5 = BULK_DIR / "expression.h5"
GENE_LIST_TXT = BULK_DIR / "gene_list.txt"
METADATA_CSV = BULK_DIR / "metadata.csv"

with h5py.File(EXPRESSION_H5, "r") as f:
    print(list(f.keys()))                      # top level
    for k in f.keys():
        o = f[k]
        if isinstance(o, h5py.Dataset):
            print(k, o.shape, o.dtype, o.chunks)
        else:
            sub = list(o.keys())
            print(k, "-> group with", len(sub), "children; first:", sub[:5])

['expression', 'genes', 'sample_ids']
expression (719432, 67186) float32 (1, 67186)
genes (67186,) object None
sample_ids (719432,) object None


In [ ]:
# ---- Subsample labelled ARCHS4 samples -> PCA -> UMAP coloured by tissue_general ----
# Assumes `samples` (indexed by sample_id, with a `tissue_general` column) exists in the notebook.
# h5 layout: expression (samples x genes), chunked one row per sample; genes; sample_ids.
from tqdm.auto import tqdm

BULK_DIR = Path("/cluster/work/boeva/rquiles/data/ARCHS4-data/data/processed/archs4")
EXPRESSION_H5 = BULK_DIR / "expression.h5"
GENE_LIST_TXT = BULK_DIR / "gene_list.txt"

N = 20_000            # number of labelled samples to subsample
SEED = 0
LABEL = "tissue_general"
USE_GENE_LIST = True  # keep only genes in gene_list.txt (smaller matrix, much less RAM)
N_HVG, N_PCS, TOP_K_LEGEND = 5000, 100, 30


def _decode(a):
    return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in a])


with h5py.File(EXPRESSION_H5, "r") as f:
    dset = f["expression"]                      # (719432, 67186), chunks (1, 67186)
    h5_ids = _decode(f["sample_ids"][:])
    h5_genes = _decode(f["genes"][:])
    assert dset.shape == (len(h5_ids), len(h5_genes))

    # genes to keep (column selection happens in memory, per row)
    if USE_GENE_LIST:
        keep_set = set(pd.read_csv(GENE_LIST_TXT, header=None)[0].astype(str))
        gmask = np.fromiter((g in keep_set for g in h5_genes), bool, len(h5_genes))
        print(f"gene_list.txt: {gmask.sum():,} of {len(keep_set):,} genes found in the h5")
        assert gmask.any(), "no overlap between gene_list.txt and h5 genes (symbol vs Ensembl?)"
    else:
        gmask = np.ones(len(h5_genes), bool)
    gidx = np.flatnonzero(gmask)

    # choose N samples that have a label
    lab = samples[LABEL].reindex(h5_ids)
    labelled = np.flatnonzero(lab.notna().to_numpy())
    print(f"{len(labelled):,} of {len(h5_ids):,} samples have a {LABEL}")
    rng = np.random.default_rng(SEED)
    idx = np.sort(rng.choice(labelled, size=min(N, len(labelled)), replace=False))

    # read row by row: each row is exactly one chunk, so this is the cheap access pattern
    X = np.empty((len(idx), len(gidx)), dtype=np.float32)
    for j, i in enumerate(tqdm(idx, desc="reading samples")):
        X[j] = dset[i, :][gidx]

obs = pd.DataFrame({LABEL: lab.iloc[idx].astype(str).to_numpy()}, index=h5_ids[idx])
adata = sc.AnnData(X, obs=obs, var=pd.DataFrame(index=h5_genes[gidx]))
adata.var_names_make_unique()
adata.obs_names_make_unique()
print(adata)

# normalisation: only if the values look like raw counts
if np.allclose(X[:200], np.round(X[:200])) and X.max() > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("normalised: CPM-like (1e4) + log1p")
else:
    print("values do not look like raw counts: used as is")

sc.pp.highly_variable_genes(adata, n_top_genes=min(N_HVG, adata.n_vars))
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS, svd_solver="arpack")
sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15)
sc.tl.umap(adata)

# collapse rare labels so the legend stays readable
top = adata.obs[LABEL].value_counts().index[:TOP_K_LEGEND]
adata.obs["label_plot"] = adata.obs[LABEL].where(adata.obs[LABEL].isin(top), "other").astype("category")

fig, ax = plt.subplots(figsize=(11, 8))
sc.pl.umap(adata, color="label_plot", ax=ax, show=False, size=4, legend_loc="right margin",
           title=f"ARCHS4 ({adata.n_obs:,} samples), PCA({N_PCS}) -> UMAP, coloured by {LABEL}")
plt.show()

gene_list.txt: 67,186 of 67,186 genes found in the h5
367,584 of 719,432 samples have a tissue_general


reading samples:   0%|          | 22/20000 [00:00<09:54, 33.60it/s]